# 40. 한국 약품 마스터 테이블 (핵심 산출물)

## 목적
모든 한국 데이터셋을 `품목일련번호`로 연결하는 crosswalk 테이블 생성.

| 컬럼 | 출처 |
|---|---|
| `item_seq` | 품목일련번호 (기본 키) |
| `item_name` | 품목명 |
| `ingr_code` | 성분코드 |
| `atc_code` | ATC 코드 |
| `has_nadal` | 낱알식별 메타데이터 존재 |
| `has_eyak` | e약은요 텍스트 존재 |
| `has_dur` | DUR 항목 존재 |
| `has_aihub_image` | AI Hub 이미지 존재 |
| `n_images` | AI Hub 이미지 수 |
| `n_dur_pairs` | DUR pair 수 |

## 결정 게이트
4종 모두 존재하는 약품이 **1,000개 이상**이면 5,000종 목표 유지;  
미만이면 만성질환 처방 subset으로 범위 축소.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
from pathlib import Path

ROOT = Path("../../").resolve()
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

In [ ]:
# 각 데이터셋 parquet 로드
available = {}

for name, path in [
    ("nadal",  INTERIM / "nadal_ident_clean.parquet"),
    ("eyak",   INTERIM / "eyak_clean.parquet"),
    ("dur",    INTERIM / "dur_clean.parquet"),
]:
    if path.exists():
        available[name] = pd.read_parquet(path)
        print(f"{name}: {available[name].shape}")
    else:
        print(f"{name}: ❌ 파일 없음 — 해당 노트북 먼저 실행 필요")

In [ ]:
# 낱알식별을 기준 테이블로 사용 (가장 포괄적인 한국 약품 목록)
if "nadal" in available:
    df_nadal = available["nadal"]

    # 품목일련번호 컬럼 정규화
    id_col_nadal = "ITEM_SEQ"
    master = df_nadal[[id_col_nadal, "ITEM_NAME"]].copy() if "ITEM_NAME" in df_nadal.columns else df_nadal[[id_col_nadal]].copy()
    master.columns = ["item_seq", "item_name"] if "ITEM_NAME" in df_nadal.columns else ["item_seq"]
    master["item_seq"] = master["item_seq"].astype(str)
    master["has_nadal"] = True
    print(f"기준 약품 수: {len(master):,}")

In [ ]:
# e약은요 join
if "eyak" in available and "master" in dir():
    df_eyak = available["eyak"]
    id_col_eyak = next((c for c in df_eyak.columns if "seq" in c.lower() or "SEQ" in c), None)
    print("e약은요 ID 컬럼 후보:", id_col_eyak)
    if id_col_eyak:
        eyak_ids = df_eyak[id_col_eyak].astype(str)
        master["has_eyak"] = master["item_seq"].isin(eyak_ids)
        print(f"e약은요 join 히트율: {master['has_eyak'].mean():.1%}")

In [ ]:
# DUR join
if "dur" in available and "master" in dir():
    df_dur = available["dur"]
    dur_id_cols = [c for c in df_dur.columns if "코드" in c or "번호" in c or "SEQ" in c.upper()]
    print("DUR ID 컬럼:", dur_id_cols)
    if dur_id_cols:
        dur_ids = pd.concat([df_dur[c].astype(str) for c in dur_id_cols[:2]]).unique()
        master["has_dur"] = master["item_seq"].isin(dur_ids)
        n_dur_map = df_dur[dur_id_cols[0]].astype(str).value_counts().to_dict()
        master["n_dur_pairs"] = master["item_seq"].map(n_dur_map).fillna(0).astype(int)
        print(f"DUR join 히트율: {master['has_dur'].mean():.1%}")

In [ ]:
# AI Hub 이미지 join (aihub_pills_summary.json에서 클래스 목록 파악)
aihub_summary_path = INTERIM / "aihub_pills_summary.json"
if aihub_summary_path.exists() and "master" in dir():
    with open(aihub_summary_path) as f:
        aihub_summary = json.load(f)
    # class_to_image_count dict가 있으면 매핑
    if "class_to_image_count" in aihub_summary:
        img_map = {str(k): v for k, v in aihub_summary["class_to_image_count"].items()}
        master["has_aihub_image"] = master["item_seq"].isin(img_map.keys())
        master["n_images"] = master["item_seq"].map(img_map).fillna(0).astype(int)
        print(f"AI Hub 이미지 hit율: {master['has_aihub_image'].mean():.1%}")
    else:
        print("aihub_pills_summary에 class_to_image_count 없음 — 20_aihub_pill_images 재실행 후 추가")
        master["has_aihub_image"] = False
        master["n_images"] = 0
else:
    if "master" in dir():
        master["has_aihub_image"] = False
        master["n_images"] = 0

In [ ]:
# ★ 결정 게이트: 4종 모두 존재하는 약품 수
if "master" in dir():
    flag_cols = [c for c in ["has_nadal", "has_eyak", "has_dur", "has_aihub_image"] if c in master.columns]
    if flag_cols:
        holy_quartet = master[flag_cols].all(axis=1)
        n_full = holy_quartet.sum()
        print(f"\n{'='*50}")
        print(f"★ 4종 데이터 완전 보유 약품: {n_full:,}개")
        print(f"  목표 1,000개 {'달성 ✓' if n_full >= 1000 else '미달 ✗ → 범위 축소 논의 필요'}")
        print(f"{'='*50}")

        # 벤 다이어그램 대용: 조합별 집계
        for col in flag_cols:
            print(f"  {col}: {master[col].sum():,}개")

In [ ]:
# 산출물 저장
if "master" in dir():
    PROCESSED.mkdir(exist_ok=True)
    master.to_parquet(PROCESSED / "korean_drug_master.parquet", index=False)
    print(f"저장: {PROCESSED / 'korean_drug_master.parquet'}")
    print(master.info())